In [3]:
import os
import pickle
import numpy as np
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List
import contextlib

# Define the Pydantic models directly here
class CustomerFeatures(BaseModel):
    tenure_months: int
    support_tickets: int
    monthly_charges: float
    monthly_active_days: int

class PredictionResponse(BaseModel):
    churn_probability: float
    predicted_class: int
    risk_level: str
    risk_explanation: str

class BatchPredictionResponse(BaseModel):
    predictions: List[PredictionResponse]

app = FastAPI(title="Churn Scoring Service", version="1.0.0")

MODEL_PATH = "model.pkl"
model = None

@contextlib.asynccontextmanager
async def lifespan(app: FastAPI):
    # Load the ML model
    global model
    if not os.path.exists(MODEL_PATH):
        raise RuntimeError(f"Model file not found at {MODEL_PATH}. Run 'python train_model.py' first.")
    with open(MODEL_PATH, "rb") as f:
        model = pickle.load(f)
    yield
    # Clean up (optional, for shutdown)

app = FastAPI(title="Churn Scoring Service", version="1.0.0", lifespan=lifespan)

def generate_explanation(features: CustomerFeatures, prob: float) -> str:
    reasons = []
    if features.support_tickets > 4:
        reasons.append("high support-ticket count")
    if features.monthly_active_days < 5:
        reasons.append("low recent app engagement")

    if prob >= 0.7:
        base = "Critical indicators present: "
    elif prob >= 0.4:
        base = "Moderate concerns: "
    else:
        return "Customer displays healthy usage and stable retention traits."

    return base + " and ".join(reasons) if reasons else "Elevated risk profile relative to baseline features."

def compute_prediction(features: CustomerFeatures) -> PredictionResponse:
    # Format properties to match training matrix structure
    input_data = np.array([[
        features.tenure_months,
        features.support_tickets,
        features.monthly_charges,
        features.monthly_active_days
    ]])

    prob = float(model.predict_proba(input_data)[0][1])
    pred_class = int(model.predict(input_data)[0])

    if prob >= 0.7:
        risk_level = "high"
    elif prob >= 0.4:
        risk_level = "medium"
    else:
        risk_level = "low"

    explanation = generate_explanation(features, prob)

    return PredictionResponse(
        churn_probability=round(prob, 2),
        predicted_class=pred_class,
        risk_level=risk_level,
        risk_explanation=explanation
    )

@app.get("/health")
def health_check():
    return {"status": "ok"}

@app.post("/predict", response_model=PredictionResponse)
def predict_single(payload: CustomerFeatures):
    try:
        return compute_prediction(payload)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/batch_predict", response_model=BatchPredictionResponse)
def predict_batch(payload: List[CustomerFeatures]):
    try:
        results = [compute_prediction(item) for item in payload]
        return BatchPredictionResponse(predictions=results)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

### Creating a Dummy Model

The FastAPI application expects a `model.pkl` file to be present. Since it's not in the environment, I'll create a simple dummy scikit-learn `LogisticRegression` model and save it to `model.pkl`. This will allow the application to start without errors.

In [4]:
from sklearn.linear_model import LogisticRegression
import pickle

# Create a dummy model (e.g., Logistic Regression)
dummy_model = LogisticRegression()

# The model needs to be fitted with some dummy data to be properly pickled
# The input features are: tenure_months, support_tickets, monthly_charges, monthly_active_days
# Let's use some example data to fit it minimally
X_dummy = [[10, 2, 50.0, 20], [5, 5, 80.0, 10], [1, 1, 30.0, 25], [12, 0, 45.0, 28]]
y_dummy = [0, 1, 0, 0] # 0 for no churn, 1 for churn

dummy_model.fit(X_dummy, y_dummy)

# Save the dummy model to 'model.pkl'
with open('model.pkl', 'wb') as f:
    pickle.dump(dummy_model, f)

print(f"Dummy model saved to {MODEL_PATH}")

Dummy model saved to model.pkl
